In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [3]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AQuAsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [6]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return float(cleaned)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            CoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Think step by step through this question. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step and correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract the option letter
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
            print(match)
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )
        
        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  0%|          | 0/205 [00:00<?, ?it/s]

<re.Match object; span=(691, 710), match='The answer is **C**'>
<re.Match object; span=(783, 798), match='The answer is E'>
<re.Match object; span=(1118, 1135), match='The answer is **C'>
<re.Match object; span=(915, 930), match='The answer is B'>


  0%|          | 1/205 [00:05<17:13,  5.06s/it]

<re.Match object; span=(1125, 1144), match='The answer is **C**'>
Accuracy: 1 / 1 = 100.00%
<re.Match object; span=(1257, 1276), match='The answer is **A**'>
<re.Match object; span=(550, 569), match='The answer is **C**'>
<re.Match object; span=(1388, 1407), match='The answer is **B**'>
<re.Match object; span=(237, 254), match='The answer is B**'>


  1%|          | 2/205 [00:06<09:29,  2.81s/it]

...

<re.Match object; span=(1411, 1428), match='The answer is **C'>
Accuracy: 159 / 191 = 83.25%
Accuracy: 160 / 192 = 83.33%
Accuracy: 160 / 193 = 82.90%
<re.Match object; span=(736, 751), match='The answer is D'>
<re.Match object; span=(836, 855), match='The answer is **B**'>
<re.Match object; span=(784, 803), match='The answer is **C**'>
<re.Match object; span=(1137, 1156), match='The answer is **E**'>
<re.Match object; span=(2249, 2264), match='The answer is C'>
<re.Match object; span=(1627, 1644), match='The answer is **B'>
<re.Match object; span=(1968, 1987), match='The answer is **B**'>
<re.Match object; span=(1592, 1611), match='The answer is **E**'>


 95%|█████████▍| 194/205 [06:21<00:19,  1.75s/it]

<re.Match object; span=(2193, 2212), match='The answer is **B**'>
Accuracy: 161 / 194 = 82.99%
Accuracy: 162 / 195 = 83.08%
Accuracy: 162 / 196 = 82.65%
Accuracy: 163 / 197 = 82.74%
Accuracy: 164 / 198 = 82.83%
Accuracy: 164 / 199 = 82.41%
Accuracy: 165 / 200 = 82.50%
Accuracy: 166 / 201 = 82.59%
Accuracy: 167 / 202 = 82.67%
Accuracy: 168 / 203 = 82.76%


100%|██████████| 205/205 [06:22<00:00,  1.87s/it]

Accuracy: 168 / 204 = 82.35%
Accuracy: 169 / 205 = 82.44%


In [4]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)


# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d.get('options', [])  # Optional if not MCQ
        correct_raw = d['correct'].strip()
        is_mcq = correct_raw.upper() in ['A', 'B', 'C', 'D', 'E']
        correct_value = correct_raw.upper() if is_mcq else float(correct_raw)

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}" if options else question

        # === Prompt Setup ===
        prompt_q = (
            Standard_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Answer the question. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Model's Answer ===
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)        
        extracted = None
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            print(match)
            if is_mcq:
                extracted = extracted_raw.upper()
            else:
                extracted = clean_and_truncate(extracted_raw)

        # === Log Block ===
        log_block = (
            f'Q: {full_question}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted Option:\n{extracted}\n'
            f'Correct:\n{correct_raw}\n\n'
        )

        # === Check Answer ===
        if is_mcq:
            is_correct = extracted == correct_value
        else:
            is_correct = extracted is not None and math.isclose(extracted, correct_value, rel_tol=1e-4)

        if is_correct:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"⚠️ Error processing entry: {d}\nTraceback:\n{traceback.format_exc()}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # === Final Summary ===
    final_summary = f"\n✅ Final Accuracy: {acc} / {total} = {acc / total:.2%}\n"
    fd.write("\n=== FINAL RESULTS ===\n" + final_summary)
    print(final_summary)


  0%|          | 0/205 [00:00<?, ?it/s]

<re.Match object; span=(327, 342), match='The answer is C'>
<re.Match object; span=(606, 621), match='The answer is B'>
<re.Match object; span=(467, 482), match='The answer is E'>
<re.Match object; span=(802, 817), match='The answer is C'>
<re.Match object; span=(695, 710), match='The answer is A'>
<re.Match object; span=(726, 745), match='The answer is **E**'>
<re.Match object; span=(126, 141), match='The answer is B'>


  0%|          | 1/205 [00:04<14:17,  4.20s/it]

<re.Match object; span=(962, 977), match='The answer is E'>
<re.Match object; span=(821, 836), match='The answer is C'>
<re.Match object; span=(589, 604), match='The answer is C'>
Accuracy: 1 / 1 = 100.00%
Accuracy: 1 / 2 = 50.00%
<re.Match object; span=(469, 484), match='The answer is C'>
<re.Match object; span=(316, 331), match='The answer is B'>
<re.Match object; span=(436, 451), match='The answer is E'>
<re.Match object; span=(1743, 1762), match='The answer is **E**'>


  1%|▏         | 3/205 [00:06<07:09,  2.12s/it]

...

<re.Match object; span=(1288, 1307), match='The answer is **E**'>
Accuracy: 167 / 200 = 83.50%
<re.Match object; span=(1313, 1328), match='The answer is B'>


 99%|█████████▊| 202/205 [05:11<00:04,  1.42s/it]

<re.Match object; span=(1262, 1281), match='The answer is **E**'>
Accuracy: 168 / 201 = 83.58%
Accuracy: 169 / 202 = 83.66%
Accuracy: 170 / 203 = 83.74%


100%|██████████| 205/205 [05:15<00:00,  1.54s/it]

<re.Match object; span=(2297, 2316), match='The answer is **B**'>
Accuracy: 171 / 204 = 83.82%
Accuracy: 172 / 205 = 83.90%

✅ Final Accuracy: 172 / 205 = 83.90%



In [5]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/AQuA/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        question = d['question']
        options = d.get('options', [])  # Optional if not MCQ
        correct_raw = d['correct'].strip()
        is_mcq = correct_raw.upper() in ['A', 'B', 'C', 'D', 'E']
        correct_value = correct_raw.upper() if is_mcq else float(correct_raw)

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}" if options else question


        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using step by step thoughts. Write your final answer as: The answer is <option letter>\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Model's Answer ===
        match = re.search(r'the answer is\s*\**([A-E])\**', ans_model, re.IGNORECASE)
        if not match:
            match = re.search(r'\b([A-E])\b', ans_model.strip()[-5:], re.IGNORECASE)        
        extracted = None
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            print(match)
            print(extracted_raw)
            if is_mcq:
                extracted = extracted_raw.upper()
            else:
                extracted = clean_and_truncate(extracted_raw)

        # === Log Block ===
        log_block = (
            f'Q: {full_question}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted Option:\n{extracted}\n'
            f'Correct:\n{correct_raw}\n\n'
        )

        # === Check Answer ===
        if is_mcq:
            is_correct = extracted == correct_value
        else:
            is_correct = extracted is not None and math.isclose(extracted, correct_value, rel_tol=1e-4)

        if is_correct:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block


    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # === Final Summary ===
    final_summary = f"\n✅ Final Accuracy: {acc} / {total} = {acc / total:.2%}\n"
    fd.write("\n=== FINAL RESULTS ===\n" + final_summary)
    print(final_summary)

    

  0%|          | 0/205 [00:00<?, ?it/s]

<re.Match object; span=(3137, 3154), match='The answer is **C'>
C


  0%|          | 1/205 [00:07<26:22,  7.76s/it]

<re.Match object; span=(2684, 2701), match='The answer is **C'>
C
Accuracy: 1 / 1 = 100.00%
<re.Match object; span=(2475, 2492), match='The answer is **B'>
B


  1%|          | 2/205 [00:12<19:40,  5.81s/it]

...

<re.Match object; span=(4218, 4235), match='The answer is **E'>
E
Accuracy: 93 / 191 = 48.69%
Accuracy: 94 / 192 = 48.96%
Accuracy: 94 / 193 = 48.70%
Accuracy: 94 / 194 = 48.45%
Accuracy: 94 / 195 = 48.21%
Accuracy: 94 / 196 = 47.96%
<re.Match object; span=(2324, 2341), match='The answer is **B'>
B


 96%|█████████▌| 197/205 [12:57<00:15,  1.96s/it]

<re.Match object; span=(2469, 2484), match='The answer is C'>
C
Accuracy: 94 / 197 = 47.72%
Accuracy: 95 / 198 = 47.98%
<re.Match object; span=(3485, 3502), match='The answer is **B'>
B


100%|██████████| 205/205 [13:06<00:00,  3.84s/it]

<re.Match object; span=(3120, 3135), match='The answer is D'>
D
Accuracy: 95 / 199 = 47.74%
Accuracy: 95 / 200 = 47.50%
Accuracy: 95 / 201 = 47.26%
Accuracy: 96 / 202 = 47.52%
Accuracy: 96 / 203 = 47.29%
Accuracy: 96 / 204 = 47.06%
Accuracy: 97 / 205 = 47.32%

✅ Final Accuracy: 97 / 205 = 47.32%

